# Lycrawl (LYCR): the 1996 approach

This is the house company for 1996. It is built the way the large search engines of 1995 and 1996 were built, and it wins 1996.

**What won in 1996 was size.** Engines competed on how many pages they had indexed, and advertised the number. The approach was simple and expensive: crawl as much of the web as the machines allow, keep every word of every page, and rank a page by how often it contains the words of the query.

Lycrawl does exactly that:

- **The crawler** is breadth-first. It takes pages in the order it finds them and fetches until the budget runs out. It obeys `robots.txt`, as the major crawlers did from 1994.
- **The ranking** is a word count. The page that contains the query's words most often comes first.

Lycrawl fails in 1998, when pages that repeat words on purpose appear, and in 2000, when quality is all that matters.

In [ ]:
from collections import Counter, deque
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit, urlunsplit
import re


def normalize(url):
    parts = urlsplit(url)
    path = parts.path or "/"
    if path != "/" and path.endswith("/"):
        path = path.rstrip("/") or "/"
    return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), path, parts.query, ""))


class Page(HTMLParser):
    """The links, the title words and the body words of one page."""

    def __init__(self, url, html):
        super().__init__()
        self.url, self.links, self.title, self.body = url, [], [], []
        self._in_title = False
        self.feed(html)

    def handle_starttag(self, tag, attrs):
        if tag == "title":
            self._in_title = True
        if tag == "a":
            for name, value in attrs:
                if name == "href" and value:
                    link = normalize(urljoin(self.url, value))
                    if urlsplit(link).scheme in ("http", "https"):
                        self.links.append(link)

    def handle_endtag(self, tag):
        if tag == "title":
            self._in_title = False

    def handle_data(self, data):
        words = re.findall(r"[a-z0-9]+", data.lower())
        (self.title if self._in_title else self.body).extend(words)


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


class Robots:
    """Fetches each host's robots.txt once, and answers whether a URL may be crawled."""

    def __init__(self, fetch):
        self.fetch = fetch
        self.rules = {}

    def allowed(self, url):
        parts = urlsplit(url)
        if parts.netloc not in self.rules:
            status, text = self.fetch(f"{parts.scheme}://{parts.netloc}/robots.txt")
            if status == 429:
                return False
            self.rules[parts.netloc] = self._disallowed(text) if status == 200 else []
        return not any(rule and parts.path.startswith(rule) for rule in self.rules[parts.netloc])

    @staticmethod
    def _disallowed(text):
        rules, applies = [], False
        for line in text.splitlines():
            key, _, value = line.partition(":")
            key, value = key.strip().lower(), value.strip()
            if key == "user-agent":
                applies = value == "*"
            elif key == "disallow" and applies:
                rules.append(value)
        return rules

## The crawler: breadth-first, polite, until the budget runs out

In [ ]:
def crawl(seed, fetch):
    robots = Robots(fetch)
    pages = {}
    seen = {normalize(seed)}
    queue = deque([normalize(seed)])
    while queue:
        url = queue.popleft()
        if not robots.allowed(url):
            continue
        status, html = fetch(url)
        if status == 429:
            break
        if status != 200:
            continue
        pages[url] = html
        for link in Page(url, html).links:
            if link not in seen:
                seen.add(link)
                queue.append(link)
    return pages

## The ranking: count the words

A page matches when it contains any word of the query. The page with the most occurrences of the query's words ranks first. This is fast, and it is easy to fool: a page that repeats "flu cure" sixty times outranks the public health unit's page about influenza.

In [ ]:
def build_engine(pages):
    counts = {}
    for url, html in pages.items():
        page = Page(url, html)
        counts[url] = Counter(page.body + page.title)

    def search(query, k=5):
        words = tokenize(query)
        scored = [(-sum(c[w] for w in words), url) for url, c in counts.items()
                  if any(c[w] for w in words)]
        return [url for _, url in sorted(scored)[:k]]

    return search